In [0]:
import pyspark.sql.functions as F

In [0]:
%sql
-- Check how many active vs historical products we have 
SELECT 
      case when end_date IS NULL THEN 'Active(end date IS NULL)' ELSE 'Expired(end date is NOT NULL)' END as status,
      count(*) AS product_count
From workspace.silver.crm_products
group by (end_date is NULL)
order by status

In [0]:
query = """ 
SELECT 
SHA2(pn.product_number,256) AS product_key, -- surrogate key
      pn.product_id,
      pn.product_number,
      pn.category_id,
      pn.product_name,
      pc.category,
      pc.subcategory,
      pc.maintenance_flag,
      pn.product_line,
      pn.product_cost,
      pn.is_cost_known,
      pn.start_date,
      CASE WHEN pn.end_date IS NULL THEN TRUE ELSE FALSE END AS is_active
FROM silver.crm_products pn
LEFT JOIN silver.erp_px_cat_g1v2 pc
ON pn.category_id = pc.category_id
"""

df =spark.sql(query)
print(f"Total products (active + Expired): {df.count()}")
print(f"Active products in dim_products:{df.filter(F.col("is_active") == True).count()}")
print(f"Expired products in dim_products:{df.filter(F.col("is_active")==False).count()}")

## Sanity checks 

In [0]:
import pyspark.sql.functions as F
print("Sample Data :")
df.limit(10).display()
print("\n Check for duplicate product keys (should be 0):")
dup_count=df.groupby("product_key").count().filter(F.col("count") > 1).count()
print(f"Duplicates : {dup_count}")

print("\n product line distribution:")
df.groupBy("product_line").count().orderBy("count",ascending=False).display()

print("\n Category Distribution:")
df.groupBy("category","subcategory").count().orderBy("category").display()
print("\n Products with unknown cost :")
df.filter(~F.col("is_cost_known")).count()

### Writing into gold table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_products")
print(f"Written {df.count()} rows to workspace.gold.dim_products")

### sanity checks

In [0]:
%sql
SELECT * FROM workspace.gold.dim_products;